# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record sets and their available field `@id`s. These `@id`s are essential for referencing each data element.

In [ ]:
# List all record sets and their fields using their @id
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record set(s):")
for i, record_set in enumerate(record_sets, 1):
    print(f"\nRecord Set {i}:")
    print(f"  @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Field @id list:")
    for field in record_set.fields:
        print(f"    - {field.id} ({field.name}, type: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract all record sets for generality and display their fields.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"\nDataFrame for Record Set '@id': {record_set.id}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select one numeric field from the main record set for demonstration.

In [ ]:
# Choose the first record set for EDA
record_set = record_sets[0]
df = dataframes[record_set.id]

# Identify numeric fields
numeric_fields = [field.id for field in record_set.fields if field.data_type in ['Number', 'Integer', 'Float'] and field.id in df.columns]
print(f"Numeric fields available in record set {record_set.id}: {numeric_fields}")
if not numeric_fields:
    raise ValueError("No numeric fields found in the chosen record set!")

# Choose the first numeric field for demonstration
numeric_field = numeric_fields[0]

# Filter for values greater than the median of the numeric field
threshold = df[numeric_field].median() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
group_fields = [field.id for field in record_set.fields if field.data_type in ['Text', 'String'] and field.id in df.columns and field.id != numeric_field]
if group_fields:
    group_field = group_fields[0]
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field:'mean_'+numeric_field})
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of '{numeric_field}' in Record Set: {record_set.id}")
plt.show()

# If a grouping field is available, add a boxplot
if 'group_field' in locals():
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides mass-spectrometry-based quantitative results for proteins from human mast cell-derived vesicles.
- We explored available record sets and fields using their `@id`.
- We demonstrated filtering, normalization, and visualization of a core numeric variable.
- Further analyses can refine feature selection and conduct advanced statistics/ML on the dataset.